<a href="https://colab.research.google.com/github/despoileralert/DSA2101-Project/blob/main/DT_Ensembles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
import time
import warnings
import os
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import cross_validate
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.decomposition import PCA, FactorAnalysis, FastICA, KernelPCA, TruncatedSVD
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import (
    AdaBoostClassifier,
    BaggingClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.feature_selection import (
    RFE,
    SelectFromModel,
    SelectKBest,
    VarianceThreshold,
    f_classif,
    mutual_info_classif,
)

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.random_projection import GaussianRandomProjection
from sklearn.tree import DecisionTreeClassifier

# Inital Setup

In [20]:
# drive mount
from google.colab import drive
drive.mount('/content/drive')

shared_folder_path = '/content/drive/MyDrive/time-series-human'

# Example: List the files inside the shared folder
import os
print(os.listdir(shared_folder_path))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['features.txt', 'activity_labels.txt', 'README.txt', 'sampler.py', 'features_info.txt', 'view.ipynb', 'RawData', 'Train', 'Model Training', 'Test', 'feature_extractor.py']


In [ ]:
df = pd.read_csv('/content/drive/MyDrive/time-series-human/Train/X_train.txt', sep=r'\s+', header=None)

new_cols = []
with open('/content/drive/MyDrive/time-series-human/features.txt', 'r') as f:
    for line in f:
        new_cols.append(line.strip())
df.columns = new_cols
df

# Cross Validation Step

In [21]:
#Firstly we handle the train test split

base = "/content/drive/MyDrive/time-series-human"

import numpy as np
from sklearn.model_selection import StratifiedGroupKFold

X = pd.read_csv(os.path.join(base, "Train", "features_train.csv")) #Train set
y = pd.read_csv(os.path.join(base, "Test", "features_test.csv"))   #Test set

X_train, X_test = X.iloc[:, 3:], y.iloc[:, 3:]
y_train, y_test = (X.iloc[:, 2]).to_numpy(), y.iloc[:, 2].to_numpy()
Xgroups, ygroups = (X.iloc[:, 0]).to_numpy(), y.iloc[:, 0].to_numpy()

print(np.unique(Xgroups))
print(np.unique(ygroups))

#Initialize CV method
sgkf = StratifiedGroupKFold(n_splits=5, shuffle = True, random_state = 42)
sgkf.get_n_splits()

groups = np.append(Xgroups, ygroups)
"""print("Xtrain is ", X_train)
print("y_train is ", y_train)"""
for i, (train_index, val_index) in enumerate(sgkf.split(X_train, y_train, Xgroups)):
    print(f"Fold {i}:")
    print(f"  Train: index length = { len(train_index)}")
    print(f"         group={np.unique(groups[train_index])}")
    print(f"  Validation:  index length = {len(val_index)}")
    print(f"         group={np.unique(groups[val_index])}")

[ 1  3  5  6  7  8 11 14 15 16 17 19 21 22 23 25 26 27 28 29 30]
[ 2  4  9 10 12 13 18 20 24]
Fold 0:
  Train: index length = 6239
         group=[ 1  3  5  6  8 11 14 15 16 17 21 22 23 26 27 29 30]
  Validation:  index length = 1528
         group=[ 7 19 25 28]
Fold 1:
  Train: index length = 6184
         group=[ 1  3  5  6  7  8 11 14 15 19 21 22 23 25 26 28 29]
  Validation:  index length = 1583
         group=[16 17 27 30]
Fold 2:
  Train: index length = 6024
         group=[ 3  5  6  7 14 16 17 19 21 22 25 26 27 28 29 30]
  Validation:  index length = 1743
         group=[ 1  8 11 15 23]
Fold 3:
  Train: index length = 6363
         group=[ 1  5  6  7  8 11 15 16 17 19 21 23 25 26 27 28 30]
  Validation:  index length = 1404
         group=[ 3 14 22 29]
Fold 4:
  Train: index length = 6258
         group=[ 1  3  7  8 11 14 15 16 17 19 22 23 25 27 28 29 30]
  Validation:  index length = 1509
         group=[ 5  6 21 26]


In [8]:
# We measure the performance of base DT and Random Foreset Models.

X_orig = X_train.loc[:, ~X_train.columns.str.startswith("wavelet")]

models = {
    "Decision tree": DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "Random forest": RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                            n_jobs=-1, random_state=42),
}

featuresets = {"X_orig": X_orig, "X_train": X_train}
results = []

# The Matthews correlation coefficient is used in machine learning as a measure of the quality of binary and multiclass classifications.
# It takes into account true and false positives and negatives and is generally regarded as a balanced measure which can be used even if the classes are of very different sizes.
# The MCC is in essence a correlation coefficient value between -1 and +1.
# A coefficient of +1 represents a perfect prediction, 0 an average random prediction and -1 an inverse prediction.
# The statistic is also known as the phi coefficient. [source: Wikipedia]

scoringmetrics = ["balanced_accuracy", "f1_macro", "matthews_corrcoef"]
#Train on original features + extended features
for feature in featuresets:
  for estname, estimator in models.items():
    s = cross_validate(estimator, featuresets[feature], y_train,
                      groups = Xgroups,
                      scoring = scoringmetrics,
                      cv = sgkf,
                      verbose = 5)
    entry = {"featureset": feature, "model": estname}
    for i in s:
      entry[i] = np.median(s[i])
    entry["model"] = estname
    results.append(entry)
  print(results)

finaltable = pd.DataFrame(results).round(3)
finaltable

[CV] END  balanced_accuracy: (test=0.688) f1_macro: (test=0.682) matthews_corrcoef: (test=0.769) total time=  10.3s
[CV] END  balanced_accuracy: (test=0.746) f1_macro: (test=0.732) matthews_corrcoef: (test=0.805) total time=   6.6s
[CV] END  balanced_accuracy: (test=0.744) f1_macro: (test=0.741) matthews_corrcoef: (test=0.853) total time=   6.1s
[CV] END  balanced_accuracy: (test=0.648) f1_macro: (test=0.652) matthews_corrcoef: (test=0.784) total time=   8.6s
[CV] END  balanced_accuracy: (test=0.620) f1_macro: (test=0.604) matthews_corrcoef: (test=0.793) total time=   9.8s


[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:   41.5s finished


[CV] END  balanced_accuracy: (test=0.764) f1_macro: (test=0.743) matthews_corrcoef: (test=0.853) total time=  40.5s
[CV] END  balanced_accuracy: (test=0.827) f1_macro: (test=0.833) matthews_corrcoef: (test=0.909) total time=  41.4s
[CV] END  balanced_accuracy: (test=0.822) f1_macro: (test=0.830) matthews_corrcoef: (test=0.950) total time=  38.1s
[CV] END  balanced_accuracy: (test=0.766) f1_macro: (test=0.763) matthews_corrcoef: (test=0.892) total time=  43.2s
[CV] END  balanced_accuracy: (test=0.745) f1_macro: (test=0.762) matthews_corrcoef: (test=0.848) total time=  41.2s
[{'featureset': 'X_orig', 'model': 'Decision tree', 'fit_time': np.float64(8.589603185653687), 'score_time': np.float64(0.011413812637329102), 'test_balanced_accuracy': np.float64(0.687828044600141), 'test_f1_macro': np.float64(0.6819081139604002), 'test_matthews_corrcoef': np.float64(0.7934058391674613)}, {'featureset': 'X_orig', 'model': 'Random forest', 'fit_time': np.float64(41.073683977127075), 'score_time': np.

[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:  3.4min finished


[CV] END  balanced_accuracy: (test=0.747) f1_macro: (test=0.723) matthews_corrcoef: (test=0.790) total time=  15.9s
[CV] END  balanced_accuracy: (test=0.760) f1_macro: (test=0.747) matthews_corrcoef: (test=0.812) total time=  12.4s
[CV] END  balanced_accuracy: (test=0.820) f1_macro: (test=0.821) matthews_corrcoef: (test=0.887) total time=  14.1s
[CV] END  balanced_accuracy: (test=0.688) f1_macro: (test=0.686) matthews_corrcoef: (test=0.756) total time=  17.1s
[CV] END  balanced_accuracy: (test=0.754) f1_macro: (test=0.739) matthews_corrcoef: (test=0.798) total time=  15.5s


[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:  1.2min finished


[CV] END  balanced_accuracy: (test=0.811) f1_macro: (test=0.803) matthews_corrcoef: (test=0.855) total time=  54.4s
[CV] END  balanced_accuracy: (test=0.836) f1_macro: (test=0.847) matthews_corrcoef: (test=0.911) total time=  59.3s
[CV] END  balanced_accuracy: (test=0.865) f1_macro: (test=0.865) matthews_corrcoef: (test=0.958) total time=  55.7s
[CV] END  balanced_accuracy: (test=0.817) f1_macro: (test=0.814) matthews_corrcoef: (test=0.894) total time=  58.4s
[CV] END  balanced_accuracy: (test=0.838) f1_macro: (test=0.820) matthews_corrcoef: (test=0.849) total time=  55.3s
[{'featureset': 'X_orig', 'model': 'Decision tree', 'fit_time': np.float64(8.589603185653687), 'score_time': np.float64(0.011413812637329102), 'test_balanced_accuracy': np.float64(0.687828044600141), 'test_f1_macro': np.float64(0.6819081139604002), 'test_matthews_corrcoef': np.float64(0.7934058391674613)}, {'featureset': 'X_orig', 'model': 'Random forest', 'fit_time': np.float64(41.073683977127075), 'score_time': np.

[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:  4.7min finished


,featureset,model,fit_time,score_time,test_balanced_accuracy,test_f1_macro,test_matthews_corrcoef
0,X_orig,Decision tree,8.590,0.011,0.688,0.682,0.793
1,X_orig,Random forest,41.074,0.119,0.766,0.763,0.892
2,X_train,Decision tree,15.451,0.014,0.754,0.739,0.798
3,X_train,Random forest,55.596,0.126,0.836,0.820,0.894


# Correlation Filter Step

In [26]:
#Now we make a correlation filter
main = pd.concat([X, y], axis=0)
correlation_matrix = main.corr()
print(main)

# Assume 'correlation_matrix' is calculated as above
# Assume 'threshold' is defined (e.g., threshold = 0.9)
threshold = 0.9
# Use absolute correlation
abs_corr_matrix = correlation_matrix.abs()

# Get the upper triangle of the correlation matrix (excluding the diagonal)
upper_triangle = abs_corr_matrix.where(np.triu(np.ones(abs_corr_matrix.shape), k=1).astype(bool))

# Find features with correlation above the threshold
to_drop = set() # Use a set to avoid duplicates
for column in upper_triangle.columns:
    highly_correlated_with = upper_triangle.index[upper_triangle[column] > threshold].tolist()
    if highly_correlated_with:
        # Basic strategy: Drop the current column if it's highly correlated with any previous one
        # More sophisticated strategies could be implemented here
        # For simplicity, let's collect the column name itself if it correlates highly
        # with any feature already checked (index < column)
        if any(upper_triangle[column] > threshold):
             to_drop.add(column) # Example: Add the second feature in the pair

print(f"Features to drop (based on threshold {threshold}): {list(to_drop)}")

# Drop the selected features from the original DataFrame
main_reduced = main.drop(columns=list(to_drop))
print(f"Original number of features: {main.shape[1]}")
print(f"Number of features after correlation filtering: {main_reduced.shape[1]}")

# Code: https://apxml.com/courses/intro-feature-engineering/chapter-6-feature-selection/filter-correlation-analysis

      user_id  experiment_id  activity_label  tBodyAcc-Mean-1  \
0           1              1               5         0.013968   
1           1              1               5        -0.008380   
2           1              1               5        -0.007727   
3           1              1               5        -0.006306   
4           1              1               5        -0.007154   
...       ...            ...             ...              ...   
3157       24             49               2         0.079284   
3158       24             49               2         0.377371   
3159       24             49               2         0.280349   
3160       24             49               2        -0.148925   
3161       24             49               2        -0.338481   

      tBodyAcc-Mean-2  tBodyAcc-Mean-3  tBodyAcc-STD-1  tBodyAcc-STD-2  \
0            0.014517        -0.107007       -0.997966       -0.990431   
1            0.056499         0.003882       -0.998279       -0.981982 